## 2026 EY AI & Data Challenge - Landsat Data Extraction Notebook

This notebook demonstrates Landsat data extraction and the creation of an output file to be used by the benchmark notebook. The baseline data is [Landsat Collection 2 Level 2](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) data from the MS Planetary Computer catalog.

## Environment preparation

In [ ]:
# !pip install uv
# !uv pip install  -r ../requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings('ignore')

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os

# Multithreading
from concurrent.futures import ThreadPoolExecutor

# Retry
import time
from pystac_client.exceptions import APIError

In [ ]:
tqdm.pandas()

In [ ]:
# Extraction
eps = 1e-10
bands_of_interest = ['green', 'blue', 'red', 'nir08', 'swir16', 'swir22']
error_row = pd.Series({band: np.nan for band in bands_of_interest})

# Paths
save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/processed/'
train_features_path = '../data/processed/landsat_features_training_200.csv'
val_features_path = '../data/processed/landsat_features_validation.csv'

# We want a ~100 m buffer around each point.  
# At the equator, 1 degree ≈ 110 km. 
# Therefore, the degree equivalent of 100 m is: buffer_deg ≈ 100 m / 110,000 m per degree ≈ 0.00089831
# This value ensures that the buffer approximately matches the pixel resolution of Landsat imagery, capturing
# a ~100 m area around each sampling location.
bbox_size = 0.00089831
bbox_for_operation = bbox_size / 2

In [ ]:
def get_bbox(row):
    '''
    Calculates a ~100m bounding box centered on the sampling coordinates.
    '''
    lat = row['Latitude']
    lon = row['Longitude']
    
    bbox = [
        lon - bbox_for_operation,
        lat - bbox_for_operation,
        lon + bbox_for_operation,
        lat + bbox_for_operation
    ]

    return bbox

In [ ]:
def get_items(catalog, row, cloud_cover_query):
    '''
    Helper function to search the STAC catalog with a specific cloud cover threshold.  
    Returns a list of found satellite scenes.
    '''
    bbox = get_bbox(row)
    
    search = catalog.search(
        collections=['landsat-c2-l2'],
        bbox=bbox,
        datetime='2011-01-01/2015-12-31',
        query={'eo:cloud_cover': {'lt': cloud_cover_query}},
    )
    return search.item_collection()

def call_api(row):
    '''
    Queries the Planetary Computer STAC API with Connection Retry Logic.
    Prevents 502 Bad Gateway errors by retrying the connection up to 3 times.
    '''
    max_retries = 3
    base_delay = 2
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # API Calls
    for attempt in range(max_retries):
        try:
            # Create catalog
            catalog = pystac_client.Client.open(
                'https://planetarycomputer.microsoft.com/api/stac/v1',
                modifier=pc.sign_inplace,
            )

            # Quality
            items = get_items(catalog, row, 10)
            if len(items) > 0: return items

            # More flexibility
            items = get_items(catalog, row, 30)
            if len(items) > 0: return items
            
            # Better than empty
            return get_items(catalog, row, 80)

        except (APIError, Exception) as e:
            if attempt == max_retries - 1:
                return [] 
            
            # Waits exponentially
            sleep_time = base_delay * (2 ** attempt)
            time.sleep(sleep_time)
            
    return []

In [ ]:
def process_bands(data):
    '''
    Computes the median value for each spectral band, handling zeros and NaNs.
    Returns a pandas Series with the extracted band values.
    '''
    results = {}

    for band in bands_of_interest:
        try:
            # Trys to convert to float
            if band in data:
                band_data = data[band].astype('float')
                
                # Compute median - ignores outliers (bords or dirty pixels)
                median_val = float(band_data.median(skipna=True).values)

                # Replace 0 with NaN
                median_val = median_val if median_val != 0 else np.nan

                # Adding to results
                results[band] = median_val
            else:
                results[band] = np.nan

        except Exception as e:
            results[band] = np.nan

    # Returns as pandas series
    return pd.Series(results)

In [2]:
def compute_Landsat_values(row):
    '''
    Main driver function:
    Retrieves available satellite scenes via API.
    Selects the scene closest to the sample date.
    Loads pixel data and extracts median band values.
    '''
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')
    bbox = get_bbox(row)
    items = call_api(row)
    
    if not items:
        print('No items to save')
        return error_row

    try:
        # Convert sample date to UTC
        sample_date_utc = date.tz_localize('UTC') if date.tzinfo is None else date.tz_convert('UTC')

        # Pick the item closest to the sample date
        items = sorted(
            items,
            key=lambda x: abs(pd.to_datetime(x.properties['datetime']).tz_convert('UTC') - sample_date_utc)
        )

        # Download pixels
        selected_item = pc.sign(items[0])

        # Load required bands
        data = stac_load([selected_item], bands=bands_of_interest, bbox=bbox).isel(time=0)
        return process_bands(data)
    
    except Exception as e:
        print(e)
        return error_row

In [ ]:
def parallel_extraction(df, max_workers=5):
    '''
    Helper function to execute feature extraction in parallel using threads.
    '''
    # Convert DataFrame to a list of dictionaries for faster iteration than native Pandas
    rows = df.to_dict('records')
    
    # 'executor.map' passes a raw dictionary, but 'compute_Landsat_values' expects a Pandas Series.
    def task_wrapper(row):
        return compute_Landsat_values(pd.Series(row))

    results = []
    print(f'Starting parallel extraction with {max_workers} workers...')
    
    # Initialize the thread pool context manager
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # executor.map distributes the tasks across threads.
        # tqdm adds a progress bar.
        results = list(tqdm(
            executor.map(task_wrapper, rows), total=len(rows)
        ))
        
    return pd.DataFrame(results)

In [ ]:
#Note

# The Landsat data extraction process for all 9,319 locations typically requires more than 7 hours when
# executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors,
# or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.

# We have already executed the full extraction for all 9,319 locations and saved the output
# to **landsat_features_training.csv**

def process_df(destination_df, original_df):
    '''
    Calculates spectral indices and merges metadata from the original dataset.
    Indices documentation: https://docs.google.com/document/d/1VOdy3sTt4UJxjBM_zrndnOT1DKKQ_yHdgqNfwF0iKn4/edit?usp=sharing

    Calculates indices:
    - NDMI (Normalized Difference Moisture Index): Measures vegetation water content. Formula: (NIR - SWIR16) / (NIR + SWIR16).
    - MNDWI (Modified Normalized Difference Water Index): Highlights open water features. Formula: (Green - SWIR16) / (Green + SWIR16).
    - NDTI (Normalized Difference Turbidity Index): Estimates water turbidity/cloudiness. Formula: (Red - Green) / (Red + Green).
    - NDSSI (Normalized Difference Suspended Sediment Index): Helps to detect thin sediments. Formula: (Blue - NIR) / (Blue + NIR).
    - Chlorophyll_Proxy: Estimates the F quantity. Formula: NIR / Green.
    - Red_Blue_Ratio: Helps to infer alcalinity. Formula: Red / Blue.
    - SI (Salinity Index): Measures the dissolved salts. Formula: np.sqrt(Blue * Red)
    '''
    # NDMI
    destination_df['NDMI'] = (
        (destination_df['nir08'] - destination_df['swir16'])
        / (destination_df['nir08'] + destination_df['swir16'] + eps)
    )
    
    # MNDWI
    destination_df['MNDWI'] = (
        (destination_df['green'] - destination_df['swir16'])
        / (destination_df['green'] + destination_df['swir16'] + eps)
    )
    
    # NDTI
    destination_df['NDTI'] = (
        (destination_df['red'] - destination_df['green'])
        / (destination_df['red'] + destination_df['green'] + eps)
    )
    
    # NDSSI
    destination_df['NDSSI'] = (
        (destination_df['blue'] - destination_df['nir08'])
        / (destination_df['blue'] + destination_df['nir08'] + eps)
    )

    # Chlorophyll_Proxy
    destination_df['Chlorophyll_Proxy'] = (
        destination_df['nir08'] / (destination_df['green'] + eps)
    )

    # Red_Blue_Ratio
    destination_df['Red_Blue_Ratio'] = (
        destination_df['red'] / (destination_df['blue'] + eps)
    )

    # Salinity Index
    destination_df['SI'] = np.sqrt(
        destination_df['blue'] * destination_df['red']
    )

    # Finalizing df
    destination_df['Latitude'] = original_df['Latitude']
    destination_df['Longitude'] = original_df['Longitude']
    destination_df['Sample Date'] = original_df['Sample Date']
    cols_to_keep = [
        'Latitude', 'Longitude', 'Sample Date', 
        'NDMI', 'MNDWI', 'NDTI', 'NDSSI', 'Chlorophyll_Proxy', 'Red_Blue_Ratio', 'SI'
    ] + bands_of_interest
    
    return destination_df[cols_to_keep]

In [ ]:
def save_df(df, table_name):
    '''
    Saves a Pandas DataFrame locally as a CSV and uploads it to a Snowflake stage.
    The file is saved temporarily to the /tmp/ directory before 
    being uploaded via the Snowflake PUT command.
    '''
    df.to_csv(f'/tmp/{table_name}.csv', index = False)
    
    session.sql(f'''
        PUT file:///tmp/{table_name}.csv
        {save_path}
        AUTO_COMPRESS=FALSE
        OVERWRITE=TRUE
    ''').collect()
    
    print(f'{table_name} saved! Refresh the browser to see the files in the sidebar')

In [ ]:
# Df for training
water_quality_df = (
    pd.read_csv('../data/raw/water_quality_training_dataset.csv')
)

# Df for testing
validation_df = (
    pd.read_csv('../data/raw/submission_template.csv')
)

## Data transformation and saving

In [ ]:
# Training
print('🚀 Running Landsat feature extraction for training data...')
landsat_train_features = parallel_extraction(water_quality_df)

print('')

# Testing
print('🚀 Running Landsat feature extraction for validation data...')
landsat_val_features = parallel_extraction(validation_df)

In [ ]:
process_df(landsat_train_features, water_quality_df)
process_df(landsat_val_features, validation_df)

## Data saving

In [ ]:
save_df(landsat_train_features, 'landsat_train_features')
save_df(landsat_val_features, 'landsat_val_features')